### 베이스라인 데이터 불러오기

In [ ]:
import os
import sys
import gc
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from functools import reduce
from dotenv import load_dotenv

import geopandas as gpd
from shapely import wkt
from IPython.display import display
from sqlalchemy import create_engine, text

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

load_dir = os.path.join("D:/seoul_bike/models_pkl/train_pkl")

try:
    X_train = joblib.load(os.path.join(load_dir, "X_train.pkl"))
    Y_train = joblib.load(os.path.join(load_dir, "Y_train.pkl"))
    X_val = joblib.load(os.path.join(load_dir, "X_val.pkl"))
    Y_val = joblib.load(os.path.join(load_dir, "Y_val.pkl"))
    X_test = joblib.load(os.path.join(load_dir, "X_test.pkl"))
    Y_test = joblib.load(os.path.join(load_dir, "Y_test.pkl"))

    print(f"========== 데이터 로드 완료 ==========")
    print(f"학습 데이터 크기: {X_train.shape}")
    print(f"검증 데이터 크기: {X_val.shape}")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {e}")

### 앙상블 베이스라인 (Voting / Stacking)


In [ ]:
# ==========================================
# 앙상블 (Voting / Stacking) — [피드백 3, 4]
# LightGBM · XGBoost · 선형모델(Ridge)을 base로, Ridge를 메타모델로 사용
# StackingRegressor(cv=TimeSeriesSplit)로 OOF 예측을 생성해 메타 입력 누수를 차단
# ==========================================

import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import VotingRegressor, StackingRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)  # 튜닝 로그 너무 많이 안 찍히게

# ==========================================
# MLflow 환경 설정
# ==========================================
USE_TEAM_SERVER = True
TEAM_SERVER_URI = "http://223.194.48.21:5000"
AUTHOR = "장수연"

mlflow.set_tracking_uri(TEAM_SERVER_URI if USE_TEAM_SERVER else "sqlite:///mlflow_seoul_bike.db")
mlflow.set_experiment("bike_demand_prediction")


# ==========================================
# 평가 지표 함수 (RMSLE 기준, 항상 원본 스케일에서 계산)
# ==========================================
def rmsle(y_true_raw, pred_raw):
    log_y = np.log1p(np.maximum(y_true_raw, 0))
    log_pred = np.log1p(np.maximum(pred_raw, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def evaluate_regr(y_true_raw, pred_raw):
    return {
        "rmsle": rmsle(y_true_raw, pred_raw),
        "rmse": np.sqrt(mean_squared_error(y_true_raw, pred_raw)),
        "mae": mean_absolute_error(y_true_raw, pred_raw)
    }

# [피드백 1] RMSLE 평가와 학습 목표를 일치시키기 위한 변환 헬퍼
def to_log1p(y_raw):
    return np.log1p(y_raw)

def to_raw(pred_log):
    # log1p 로 학습했으므로 예측은 expm1 로 복원, 음수는 0으로 clip
    return np.clip(np.expm1(pred_log), 0, None)

print("\n========== 앙상블 모델 학습 시작 ==========")
ENSEMBLE_BASE_MODELS = ["LightGBM", "XGBoost", "Ridge"]

def build_model(model_name, trial, SEED=42):
    if model_name == "LinearRegression":
        return make_pipeline(StandardScaler(), LinearRegression())

    if model_name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

    if model_name == "Lasso":
        return make_pipeline(StandardScaler(), Lasso(alpha=0.1))

    if model_name == "ElasticNet":
        return make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5))

    if model_name == "RandomForest":
        return RandomForestRegressor(n_estimators=50, max_depth=10, random_state=SEED, n_jobs=-1)

    if model_name == "GradientBoosting":
        return GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED)

    if model_name == "LightGBM":
        return LGBMRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, verbosity=-1)

    if model_name == "XGBoost":
        return XGBRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, tree_method='hist')

    raise ValueError(f"알 수 없는 모델명: {model_name}")

MODEL_NAMES = [
    "LinearRegression", "Ridge", "Lasso", "ElasticNet",
    "RandomForest", "GradientBoosting", "LightGBM", "XGBoost"
]

N_TRIALS = 30       # 모델별 Optuna 탐색 횟수 (필요에 맞게 조절하세요)
N_SPLITS = 5        # [피드백 2] TimeSeriesSplit 폴드 수
tscv = TimeSeriesSplit(n_splits=N_SPLITS)


# ==========================================
# 모델 학습(Optuna + TimeSeriesSplit 튜닝 포함) 및 MLflow 기록
# ==========================================
results = []
tuned_models = {}        # {target_name: {model_name: fitted_model}}
best_params_store = {}   # {target_name: {model_name: best_params}}
run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

TARGET_COLUMNS = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']

for target_name in TARGET_COLUMNS:
    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]
    y_train_log = to_log1p(y_train_raw)

    base_estimators = []
    for m_name in ENSEMBLE_BASE_MODELS:
        if m_name not in best_params_store[target_name]:
            continue
        # ENSEMBLE_BASE_MODELS = ["LightGBM", "XGBoost", "Ridge"] 이므로
        # LinearRegression 분기는 절대 실행되지 않아 제거하고, 튜닝된 파라미터로 바로 생성
        fixed_trial = optuna.trial.FixedTrial(best_params_store[target_name][m_name])
        fresh_model = build_model(m_name, fixed_trial)
        # Voting/Stacking이 이미 n_jobs=-1로 병렬 처리하므로,
        # 베이스 모델 자체의 병렬처리는 1로 낮춰 중첩 병렬(oversubscription)을 방지
        if "n_jobs" in fresh_model.get_params():
            fresh_model.set_params(n_jobs=1)
        base_estimators.append((m_name, fresh_model))

    if len(base_estimators) < 2:
        print(f"[{target_name}] 앙상블 base 모델이 부족해 스킵합니다.")
        continue

    # ---- Voting: 기준선 ----
    voting_model = VotingRegressor(estimators=base_estimators, n_jobs=-1)
    voting_model.fit(X_train, y_train_log)
    voting_pred_raw = to_raw(voting_model.predict(X_val))
    voting_metrics = evaluate_regr(y_val_raw.values, voting_pred_raw)

    with mlflow.start_run(run_name=f"Voting_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Voting")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(voting_metrics)

    results.append({
        "target": target_name, "model_name": "Voting",
        "rmsle": voting_metrics["rmsle"], "rmse": voting_metrics["rmse"], "mae": voting_metrics["mae"]
    })

    # ---- Stacking: OOF 예측 기반 확장 (누수 차단) ----
    stacking_model = StackingRegressor(
        estimators=base_estimators,
        final_estimator=Ridge(),
        cv=TimeSeriesSplit(n_splits=N_SPLITS),   # [피드백 4] OOF 예측 생성 → 누수 차단
        n_jobs=-1
    )
    stacking_model.fit(X_train, y_train_log)
    stacking_pred_raw = to_raw(stacking_model.predict(X_val))
    stacking_metrics = evaluate_regr(y_val_raw.values, stacking_pred_raw)

    with mlflow.start_run(run_name=f"Stacking_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Stacking")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(stacking_metrics)

    results.append({
        "target": target_name, "model_name": "Stacking",
        "rmsle": stacking_metrics["rmsle"], "rmse": stacking_metrics["rmse"], "mae": stacking_metrics["mae"]
    })

print("========== 앙상블 모델 학습 완료 ==========")

results_df = pd.DataFrame(results)
results_df.rename(columns={'rmsle': 'RMSLE', 'rmse': 'RMSE', 'mae': 'MAE'}, inplace=True)
display(results_df.sort_values(by=['target', 'RMSLE']))
